# Tone Conversion Demo: Part 1 (PyTorch) → Part 2 (ONNX)

This notebook walks through:
- **Part 1**: Build reference SE with OpenVoice v2 (PyTorch), save portable format; optionally create base_se and export converter to ONNX.
- **Part 2**: Run MeloTTS ONNX + tone conversion (no PyTorch required at inference).

Part 1 requires: PyTorch, openvoice (openvoice_v2), librosa.  
Part 2 requires: onnxruntime, numpy, soundfile (and the files produced in Part 1).

---
## Part 1 – Build reference SE (OpenVoice v2, PyTorch)

### 1.1 Paths and setup

Set `OPENVOICE_V2_DIR` to the path of the openvoice_v2 folder (so we can import openvoice and use checkpoints_v2). If you run the notebook from MeloTTS_lite, use `../openvoice_v2`.

In [1]:
import os
import sys
import json

# Path to openvoice_v2 (for Part 1)
OPENVOICE_V2_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", "openvoice_v2"))
CKPT_CONVERTER = os.path.join(OPENVOICE_V2_DIR, "checkpoints_v2", "converter")

# Outputs for Part 1 (we'll save SE and converter here for Part 2)
MELOTTS_LITE_DIR = os.getcwd()  # assume notebook is run from MeloTTS_lite
OUTPUT_DIR = os.path.join(MELOTTS_LITE_DIR, "tone_conversion_outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)

TARGET_SE_JSON = os.path.join(OUTPUT_DIR, "target_se.json")
BASE_SE_JSON = os.path.join(OUTPUT_DIR, "base_se.json")
CONVERTER_ONNX = os.path.join(OUTPUT_DIR, "converter.onnx")
CONVERTER_CONFIG = os.path.join(OUTPUT_DIR, "converter_config.json")

print("OpenVoice v2 dir:", OPENVOICE_V2_DIR)
print("Converter checkpoint:", CKPT_CONVERTER)
print("Output dir:", OUTPUT_DIR)

OpenVoice v2 dir: c:\Users\CailenRobertson\Documents\VSworkspace\ET_Studios\VoiceGeneration\openvoice_v2
Converter checkpoint: c:\Users\CailenRobertson\Documents\VSworkspace\ET_Studios\VoiceGeneration\openvoice_v2\checkpoints_v2\converter
Output dir: c:\Users\CailenRobertson\Documents\VSworkspace\ET_Studios\VoiceGeneration\MeloTTS_lite\tone_conversion_outputs


### 1.2 Load OpenVoice v2 ToneColorConverter and extract target SE

Replace `REFERENCE_VOICE_WAV` with a path to your reference voice recording (WAV/MP3).

In [2]:
if OPENVOICE_V2_DIR not in sys.path:
    sys.path.insert(0, OPENVOICE_V2_DIR)

import torch
from openvoice.api import ToneColorConverter

REFERENCE_VOICE_WAV = os.path.join(OUTPUT_DIR, "Jack_Hughman.mp3")  # replace with your file

if not os.path.isfile(REFERENCE_VOICE_WAV):
    print("No reference voice at", REFERENCE_VOICE_WAV)
    print("Generate a short recording or use an existing WAV and set REFERENCE_VOICE_WAV.")
else:
    device = "cuda:0" if torch.cuda.is_available() else "cpu"
    config_path = os.path.join(CKPT_CONVERTER, "config.json")
    ckpt_path = os.path.join(CKPT_CONVERTER, "checkpoint.pth")
    
    tone_color_converter = ToneColorConverter(config_path, device=device, enable_watermark=False)
    tone_color_converter.load_ckpt(ckpt_path)
    
    target_se = tone_color_converter.extract_se(REFERENCE_VOICE_WAV, se_save_path=None)
    se_1d = target_se.cpu().float().numpy().reshape(-1)
    data = {"se": se_1d.tolist(), "gin_channels": len(se_1d), "version": "v2"}
    with open(TARGET_SE_JSON, "w", encoding="utf-8") as f:
        json.dump(data, f, separators=(",", ":"))
    print("Saved target SE to", TARGET_SE_JSON)

c:\Users\CailenRobertson\.conda\envs\black6\Lib\site-packages\torch\nn\utils\weight_norm.py:143: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
c:\Users\CailenRobertson\Documents\VSworkspace\ET_Studios\VoiceGeneration\openvoice_v2\openvoice\api.py:36: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serializ

Loaded checkpoint 'c:\Users\CailenRobertson\Documents\VSworkspace\ET_Studios\VoiceGeneration\openvoice_v2\checkpoints_v2\converter\checkpoint.pth'
missing/unexpected keys: [] []
Saved target SE to c:\Users\CailenRobertson\Documents\VSworkspace\ET_Studios\VoiceGeneration\MeloTTS_lite\tone_conversion_outputs\target_se.json


c:\Users\CailenRobertson\.conda\envs\black6\Lib\site-packages\torch\functional.py:704: UserWarning: stft with return_complex=False is deprecated. In a future pytorch release, stft will return complex tensors for all inputs, and return_complex=False will raise an error.
Note: you can still call torch.view_as_real on the complex output to recover the old return format. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\SpectralOps.cpp:878.)
  return _VF.stft(  # type: ignore[attr-defined]


### 1.3 Create base_se (MeloTTS default speaker)

Generate a short WAV with MeloTTS ONNX, then extract its SE so the converter knows the "source" voice. Run this cell only if you don't already have `base_se.json`.

In [10]:
# Generate base speaker WAV with MeloTTS ONNX (Part 2 dependency, but we need the WAV for Part 1)
BASE_SPEAKER_WAV = os.path.join(OUTPUT_DIR, "base_speaker.wav")

# from tts_onnx import TTS
from melo.api import TTS
device = "auto"
# Speed is adjustable
speed = 1.0

# English 
text = "Complete line-up checks. When ready for takeoff, commence engine run-up checks and takeoff procedures"
model = TTS(language='EN', device=device)
speaker_ids = model.hps.data.spk2id
# American accent
output_path = 'en-us.wav'
model.tts_to_file(text, speaker_ids['EN-US'], BASE_SPEAKER_WAV, speed=speed)


Complete line-up checks. When ready for takeoff, commence engine run-up checks and takeoff procedures
 > ===========================


100%|██████████| 1/1 [00:00<00:00,  2.71it/s]


In [ ]:


# tts = TTS(model_path=os.path.join(MELOTTS_LITE_DIR, "model_en.onnx"))
# text_base = "This is a sample of the base speaker voice for tone conversion."
# tts.synthesize(text_base, output_path=BASE_SPEAKER_WAV, speed=1.0)
# print("Generated", BASE_SPEAKER_WAV)

# Extract base SE with OpenVoice v2
if OPENVOICE_V2_DIR not in sys.path:
    sys.path.insert(0, OPENVOICE_V2_DIR)
from openvoice.api import ToneColorConverter

device = "cuda:0" if torch.cuda.is_available() else "cpu"
config_path = os.path.join(CKPT_CONVERTER, "config.json")
ckpt_path = os.path.join(CKPT_CONVERTER, "checkpoint.pth")
tone_color_converter = ToneColorConverter(config_path, device=device, enable_watermark=False)
tone_color_converter.load_ckpt(ckpt_path)
source_se = torch.load(f'{OPENVOICE_V2_DIR}/checkpoints_v2/base_speakers/ses/en_au.pth', map_location=device)

OUTPUT_CONVERTED_VOICE_WAV = os.path.join(OUTPUT_DIR, "output.wav")  # replace with your file

base_se = tone_color_converter.extract_se(BASE_SPEAKER_WAV, se_save_path=None)
tone_color_converter.convert(
        audio_src_path=BASE_SPEAKER_WAV, 
        src_se=source_se, 
        tgt_se=target_se, 
        output_path=OUTPUT_CONVERTED_VOICE_WAV,
        message="black6")
se_1d = base_se.cpu().float().numpy().reshape(-1)
data = {"se": se_1d.tolist(), "gin_channels": len(se_1d), "version": "v2"}
with open(BASE_SE_JSON, "w", encoding="utf-8") as f:
    json.dump(data, f, separators=(",", ":"))
print("Saved base SE to", BASE_SE_JSON)

Loaded checkpoint 'c:\Users\CailenRobertson\Documents\VSworkspace\ET_Studios\VoiceGeneration\openvoice_v2\checkpoints_v2\converter\checkpoint.pth'
missing/unexpected keys: [] []
Saved base SE to c:\Users\CailenRobertson\Documents\VSworkspace\ET_Studios\VoiceGeneration\MeloTTS_lite\tone_conversion_outputs\base_se.json


### 1.4 Export converter to ONNX

Run once per converter version. Exports `converter.onnx` and `converter_config.json` to OUTPUT_DIR.

In [8]:
if OPENVOICE_V2_DIR not in sys.path:
    sys.path.insert(0, OPENVOICE_V2_DIR)

from openvoice.api import ToneColorConverter
from openvoice.utils import get_hparams_from_file


class VoiceConversionWrapper(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, spec, spec_lengths, sid_src, sid_tgt, tau):
        o_hat, _, _ = self.model.voice_conversion(
            spec, spec_lengths, sid_src=sid_src, sid_tgt=sid_tgt, tau=tau
        )
        return o_hat


config_path = os.path.join(CKPT_CONVERTER, "config.json")
ckpt_path = os.path.join(CKPT_CONVERTER, "checkpoint.pth")
converter = ToneColorConverter(config_path, device="cpu", enable_watermark=False)
converter.load_ckpt(ckpt_path)
wrapper = VoiceConversionWrapper(converter.model)
wrapper.eval()

hps = converter.hps
spec_channels = hps.data.filter_length // 2 + 1
gin_channels = hps.model.gin_channels
T, batch = 200, 1

spec = torch.randn(batch, spec_channels, T, dtype=torch.float32)
spec_lengths = torch.tensor([T], dtype=torch.int64)
sid_src = torch.randn(batch, gin_channels, 1, dtype=torch.float32)
sid_tgt = torch.randn(batch, gin_channels, 1, dtype=torch.float32)
tau = torch.tensor([0.3], dtype=torch.float32)

torch.onnx.export(
    wrapper,
    (spec, spec_lengths, sid_src, sid_tgt, tau),
    CONVERTER_ONNX,
    opset_version=14,
    input_names=["spec", "spec_lengths", "sid_src", "sid_tgt", "tau"],
    output_names=["audio"],
    dynamic_axes={
        "spec": {0: "batch", 2: "time"},
        "spec_lengths": {0: "batch"},
        "audio": {0: "batch", 2: "audio_time"},
    },
)
print("Exported", CONVERTER_ONNX)

config_export = {
    "filter_length": int(hps.data.filter_length),
    "sampling_rate": int(hps.data.sampling_rate),
    "hop_length": int(hps.data.hop_length),
    "win_length": int(hps.data.win_length),
    "gin_channels": int(gin_channels),
    "spec_channels": int(spec_channels),
    "version": "v2",
}
with open(CONVERTER_CONFIG, "w", encoding="utf-8") as f:
    json.dump(config_export, f, indent=2)
print("Wrote", CONVERTER_CONFIG)

Loaded checkpoint 'c:\Users\CailenRobertson\Documents\VSworkspace\ET_Studios\VoiceGeneration\openvoice_v2\checkpoints_v2\converter\checkpoint.pth'
missing/unexpected keys: [] []
Exported c:\Users\CailenRobertson\Documents\VSworkspace\ET_Studios\VoiceGeneration\MeloTTS_lite\tone_conversion_outputs\converter.onnx
Wrote c:\Users\CailenRobertson\Documents\VSworkspace\ET_Studios\VoiceGeneration\MeloTTS_lite\tone_conversion_outputs\converter_config.json


---
## Part 2 – TTS + tone conversion (ONNX only)

No PyTorch required. Uses: `tts_onnx`, `tone_converter_onnx`, and the files from Part 1 (`base_se.json`, `target_se.json`, `converter.onnx`, `converter_config.json`).

### 2.1 Explicit steps: synthesize → convert → save

In [5]:
import os
import sys
import json

# Path to openvoice_v2 (for Part 1)
OPENVOICE_V2_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", "openvoice_v2"))
CKPT_CONVERTER = os.path.join(OPENVOICE_V2_DIR, "checkpoints_v2", "converter")

# Outputs for Part 1 (we'll save SE and converter here for Part 2)
MELOTTS_LITE_DIR = os.getcwd()  # assume notebook is run from MeloTTS_lite
OUTPUT_DIR = os.path.join(MELOTTS_LITE_DIR, "tone_conversion_outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)

TARGET_SE_JSON = os.path.join(OUTPUT_DIR, "target_se_andrew.json")
BASE_SE_JSON = os.path.join(OUTPUT_DIR, "base_se.json")
CONVERTER_ONNX = os.path.join(OUTPUT_DIR, "converter.onnx")
CONVERTER_CONFIG = os.path.join(OUTPUT_DIR, "converter_config.json")

print("OpenVoice v2 dir:", OPENVOICE_V2_DIR)
print("Converter checkpoint:", CKPT_CONVERTER)
print("Output dir:", OUTPUT_DIR)

OpenVoice v2 dir: c:\Users\CailenRobertson\Documents\VSworkspace\ET_Studios\VoiceGeneration\openvoice_v2
Converter checkpoint: c:\Users\CailenRobertson\Documents\VSworkspace\ET_Studios\VoiceGeneration\openvoice_v2\checkpoints_v2\converter
Output dir: c:\Users\CailenRobertson\Documents\VSworkspace\ET_Studios\VoiceGeneration\MeloTTS_lite\tone_conversion_outputs


In [6]:
from tts_onnx import TTS
from tone_converter_onnx import (
    load_se_from_file,
    OnnxToneConverter,
    apply_tone_conversion,
)
import soundfile as sf

model_path = os.path.join(MELOTTS_LITE_DIR, "model_en.onnx")
tts = TTS(model_path=model_path)

text = "Complete line-up checks. When ready for takeoff, commence engine run-up checks and takeoff procedures."
audio, sr = tts.synthesize(text, output_path=None, speed=1.0)
print("TTS output: shape", audio.shape, "sr", sr)

converter = OnnxToneConverter(CONVERTER_ONNX, config_path=CONVERTER_CONFIG)
converted, out_sr = apply_tone_conversion(
    audio, sr, BASE_SE_JSON, TARGET_SE_JSON, converter=converter, tau=0.3
)

out_path = os.path.join(OUTPUT_DIR, "output_explicit.wav")
sf.write(out_path, converted, out_sr)
print("Saved", out_path)

TTS output: shape (273408,) sr 44100
Saved c:\Users\CailenRobertson\Documents\VSworkspace\ET_Studios\VoiceGeneration\MeloTTS_lite\tone_conversion_outputs\output_explicit.wav


### 2.2 Convenience: single call

In [4]:
from tone_converter_onnx import synthesize_then_convert

out_path2 = os.path.join(OUTPUT_DIR, "output_convenience.wav")
synthesize_then_convert(
    "Complete line-up checks. When ready for takeoff, commence engine run-up checks and takeoff procedures",
    out_path2,
    target_se_path=TARGET_SE_JSON,
    converter_onnx_path=CONVERTER_ONNX,
    converter_config_path=CONVERTER_CONFIG,
    src_se_path=BASE_SE_JSON,
    model_path=model_path,
    tau=0.3,
)
print("Saved", out_path2)

Saved c:\Users\CailenRobertson\Documents\VSworkspace\ET_Studios\VoiceGeneration\MeloTTS_lite\tone_conversion_outputs\output_convenience.wav


### 2.3 Optional: play back (if in Jupyter with audio widget)

In [ ]:
try:
    from IPython.display import Audio
    display(Audio(converted, rate=out_sr))
except Exception as e:
    print("Playback skipped:", e)